# TP: Text Generation with a Large Language Model

_From [Introduction to Deep Learning, TP11, S. Van Gool](https://www.samvangool.net/old/teaching/iap/)_

In this practical, you will implement several text generation strategies using a pretrained language model. We use the [transformers](https://huggingface.co/docs/transformers/) library from Hugging Face and OpenAI's [GPT-2 Small](https://huggingface.co/gpt2) model (124M parameters).

**Useful links:**
- [GPT2LMHeadModel](https://huggingface.co/docs/transformers/model_doc/gpt2#transformers.GPT2LMHeadModel)
- [GPT2Tokenizer](https://huggingface.co/docs/transformers/model_doc/gpt2#transformers.GPT2Tokenizer)
- [`model.generate()` documentation](https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
- [How to generate text (Hugging Face blog)](https://huggingface.co/blog/how-to-generate)

## Setup

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch
import torch.nn.functional as F

device = (
    "cuda:0"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Using device: {device}")

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2', pad_token_id=tokenizer.eos_token_id).to(device)
model.eval()
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## Tokenizer

Useful functions:
- `tokenizer.encode(text)` — encode a string into a list of token ids
- `tokenizer.decode(ids)` — decode token ids back to a string
- `tokenizer.convert_ids_to_tokens(ids)` — convert ids to token strings
- `tokenizer.get_vocab()` — returns the full vocabulary as a Python dict

Note: the character `Ġ` represents a space ([more info](https://huggingface.co/docs/transformers/tokenizer_summary)).

**TODO:** Explore the vocabulary.

1. How many tokens does the vocabulary contain?
2. What is the id of the token `hello`? And `Hello`?
3. What is the token with the highest index?
4. What is the token at index 1042?

In [ ]:
vocab = tokenizer.get_vocab()

# TODO: answer the questions above

**TODO:** Tokenize the following strings and observe the results. What do you notice?

```
It was the best of times, it was the worst of times,
it was the age of wisdom, it was the age of foolishness
```

```
C'était le meilleur des temps, c'était le pire des temps ;
c'était l'âge de la sagesse, c'était l'âge de la folie
```

```
3592 * 12046 = 43269232
```

**Hints:**
- Use `tokenizer.encode()` to get ids, then `tokenizer.convert_ids_to_tokens()` to see the subwords
- Compare the number of tokens for English vs French vs arithmetic

In [ ]:
# TODO: tokenize the three strings and display the tokens

## Logits

The following code tokenizes a text into a PyTorch tensor and passes it through the model:

```python
tokens = tokenizer(text, return_tensors="pt").to(device)
outputs = model(**tokens)
```

The `outputs` object contains many fields (attentions, loss, ...), but we focus on `outputs.logits`.

**TODO:** Pass a text of N tokens through the model.

1. What is the shape of `outputs.logits`?
2. What does the entry at position $(i, j)$ represent?

In [ ]:
text = "The meaning of life is"

# TODO: tokenize, pass through the model, inspect logits shape

**TODO:** For each token position, find the 10 most likely next tokens and examine their probabilities.

**Hints:**
- Use [`torch.topk`](https://pytorch.org/docs/stable/generated/torch.topk.html) to get the top-k values and indices
- Use [`F.softmax`](https://pytorch.org/docs/stable/generated/torch.nn.functional.softmax.html) to convert logits to probabilities
- Use `tokenizer.convert_ids_to_tokens()` to display readable tokens

In [ ]:
# TODO: for each position, show the top-10 predictions with their probabilities

## Greedy Search

The simplest generation strategy: at each step, pick the token with the highest probability.

**TODO:** Write a function `greedy` that takes a list of token ids and an integer `length`, and iteratively appends the most probable next token for `length` steps.

**Hints:**
- At each step, pass the current sequence through the model
- Take `outputs.logits[0, -1]` to get logits for the last position
- Use `argmax` to pick the most probable token

In [ ]:
@torch.no_grad()
def greedy(model, tokens, length):
    """Generate text using greedy search."""
    tokens = list(tokens)
    generated_probs = []
    for _ in range(length):
        input_ids = torch.tensor([tokens], device=device)
        # TODO: get logits, find the most probable next token, append it

    return tokens, generated_probs

Test your function on a few prompts and observe the probability of each generated token.

In [ ]:
prompts = [
    "The meaning of life is",
    "Once upon a time",
    "The president of France",
]

for prompt in prompts:
    input_ids = tokenizer.encode(prompt)
    output_ids, probs = greedy(model, input_ids, length=30)
    print(f"Prompt: {prompt!r}")
    print(f"Output: {tokenizer.decode(output_ids)!r}")
    print(f"Probs:  {['%.2f' % p for p in probs]}")
    print()

## Top-k Sampling and Temperature

In **top-k sampling**, only the $k$ highest-ranked tokens are considered: all other logits are set to $-\infty$. Then softmax is applied and we **sample** from the resulting distribution.

**TODO:** Implement a function `top_k_sampling(tokens, length, k)` that generates text using top-k sampling.

**Hints:**
- Use [`torch.topk`](https://pytorch.org/docs/stable/generated/torch.topk.html) to find the top-k logits
- Set all other logits to `float("-inf")`
- Use [`torch.multinomial`](https://pytorch.org/docs/stable/generated/torch.multinomial.html) to sample from the distribution

In [ ]:
@torch.no_grad()
def top_k_sampling(model, tokens, length, k=50, temperature=1.0):
    """Generate text using top-k sampling with temperature."""
    tokens = list(tokens)
    generated_probs = []
    for _ in range(length):
        input_ids = torch.tensor([tokens], device=device)
        # TODO: get logits, filter to top-k, sample

    return tokens, generated_probs

Test on the same prompts as before and observe the probabilities. What do you notice compared to greedy search?

In [ ]:
for prompt in prompts:
    input_ids = tokenizer.encode(prompt)
    output_ids, probs = top_k_sampling(model, input_ids, length=30, k=50)
    print(f"Prompt: {prompt!r}")
    print(f"Output: {tokenizer.decode(output_ids)!r}")
    print(f"Probs:  {['%.2f' % p for p in probs]}")
    print()

**TODO:** Add a `temperature` parameter to your function. Before applying softmax, divide the logits by `temperature`.

$$p_i = \text{softmax}\left(\frac{z_i}{T}\right)$$

- $T \to 0$: the distribution becomes peaked (approaches greedy)
- $T = 1$: unchanged
- $T > 1$: the distribution becomes flatter (more random)

Experiment with temperatures close to 0 and equal to 2.

In [ ]:
prompt = "The meaning of life is"
input_ids = tokenizer.encode(prompt)

for temp in [0.1, 0.7, 1.0, 2.0]:
    output_ids, probs = top_k_sampling(model, input_ids, length=30, k=50, temperature=temp)
    print(f"Temperature {temp}:")
    print(f"  {tokenizer.decode(output_ids)!r}")
    print()

## Top-p (Nucleus) Sampling

**Top-p sampling** selects the smallest set of tokens whose cumulative probability reaches at least $p$, applies softmax, and samples from the resulting distribution. This method is also called **nucleus sampling** ([Holtzman et al., 2019](https://arxiv.org/abs/1904.09751)).

Unlike top-k (fixed number of candidates), top-p adapts: when the model is confident, fewer tokens are kept; when uncertain, more tokens are considered.

**TODO:** Implement `top_p_sampling(tokens, length, p, temperature)`.

**Hints:**
- Sort logits in descending order with [`torch.sort`](https://pytorch.org/docs/stable/generated/torch.sort.html)
- Compute cumulative probabilities with [`torch.cumsum`](https://pytorch.org/docs/stable/generated/torch.cumsum.html)
- Mask out tokens where the cumulative probability (excluding the current token) exceeds $p$

In [ ]:
@torch.no_grad()
def top_p_sampling(model, tokens, length, p=0.9, temperature=1.0):
    """Generate text using top-p (nucleus) sampling."""
    tokens = list(tokens)
    generated_probs = []
    for _ in range(length):
        input_ids = torch.tensor([tokens], device=device)
        # TODO: get logits, filter by cumulative probability, sample

    return tokens, generated_probs

Test and compare with the other methods.

In [ ]:
prompt = "The meaning of life is"
input_ids = tokenizer.encode(prompt)

for p_val in [0.5, 0.9, 0.95]:
    output_ids, probs = top_p_sampling(model, input_ids, length=30, p=p_val)
    print(f"Top-p = {p_val}:")
    print(f"  {tokenizer.decode(output_ids)!r}")
    print()

## Beam Search

**Beam search** keeps track of the `beams` most probable sequences at each step, and recursively generates their continuations up to `length` tokens. Each sequence is scored by the sum of log-probabilities of the chosen tokens. The sequence with the best final score is returned.

Unlike sampling methods, beam search is **deterministic** and optimizes for overall sequence probability rather than making locally optimal choices.

**TODO:** Implement `beam_search(tokens, length, beams)`.

**Hints:**
- Maintain a list of `(sequence, cumulative_log_prob)` candidates
- At each step, expand each candidate with its top-`beams` next tokens
- Keep only the overall top-`beams` candidates
- Use [`F.log_softmax`](https://pytorch.org/docs/stable/generated/torch.nn.functional.log_softmax.html) for numerical stability

In [ ]:
@torch.no_grad()
def beam_search(model, tokens, length, beams=5):
    """Generate text using beam search."""
    # Each candidate: (token_list, cumulative_log_prob)
    candidates = [(list(tokens), 0.0)]

    for _ in range(length):
        all_next = []
        for seq, score in candidates:
            input_ids = torch.tensor([seq], device=device)
            # TODO: expand each candidate with its top-beams next tokens

        # TODO: keep only the top beams candidates
        candidates = all_next[:beams]

    best_seq, best_score = candidates[0]
    return best_seq, best_score

Test and compare with the other strategies.

In [ ]:
prompt = "The meaning of life is"
input_ids = tokenizer.encode(prompt)

for n_beams in [1, 3, 5, 10]:
    output_ids, score = beam_search(model, input_ids, length=30, beams=n_beams)
    print(f"Beams = {n_beams:2d} (score: {score:.2f}):")
    print(f"  {tokenizer.decode(output_ids)!r}")
    print()

## Comparison with `model.generate()`

All the strategies above are also implemented in the [`model.generate()`](https://huggingface.co/docs/transformers/main/en/main_classes/text_generation) function from the `transformers` library. See also: [How to generate text (Hugging Face blog)](https://huggingface.co/blog/how-to-generate).

```python
input_ids = tokenizer.encode("The meaning of life is", return_tensors="pt").to(device)

# Greedy
model.generate(input_ids, max_new_tokens=50, do_sample=False)

# Top-k sampling with temperature
model.generate(input_ids, max_new_tokens=50, do_sample=True, top_k=50, temperature=0.7)

# Top-p (nucleus) sampling
model.generate(input_ids, max_new_tokens=50, do_sample=True, top_p=0.9)

# Beam search
model.generate(input_ids, max_new_tokens=50, num_beams=5, early_stopping=True)
```

Verify that your implementations match the library's output.

In [ ]:
# TODO: compare your implementations with model.generate()

## La Disparition

In 1969, Georges Perec wrote [*La Disparition*](https://en.wikipedia.org/wiki/A_Void), an entire novel without ever using the letter **e** — the most common letter in French (and English). This constrained writing technique is called a **lipogram**.

**TODO:** Define a wrapper function that masks out all tokens containing the letter `e` (or `E`). Since your generation functions take `model` as a parameter, you can simply pass this wrapper to reuse all your existing code.

**Hints:**
- Build a boolean mask of all token ids whose decoded string contains `e` or `E`
- Write a wrapper function that calls the real model, then sets `logits[forbidden_mask] = -inf` before returning
- The wrapper should return the same type of object as the original model (with a `.logits` attribute)
- Run all four strategies (greedy, top-k, top-p, beam search) with the wrapper and compare the outputs

In [ ]:
# TODO: build the forbidden-token mask, define a wrapper model, and compare all strategies